In [ ]:
import numpy as np
import pandas as pd

data = pd.read_csv('/content/data_3_2.csv').to_numpy()
x = data[:, :-1]
y = data[:, -1]

In [ ]:
#Feature Scaling
x_scaled = (x - np.mean(x, axis=0)) / np.std(x, axis=0)

In [ ]:
import torch
import torch.nn as nn

class MyModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.linear = nn.Linear(2, 1)

  def forward(self, x):
    output = self.linear(x) #tensor(m, 1)
    output = output.squeeze(1) #tensor(m)
    output = torch.sigmoid(output)
    return output

In [ ]:
def create_train_model(tx, ty, num_iter=50):
  model = MyModel()
  opt = optim.Adam(model.parameters(), lr=1)
  cost_func = nn.BCELoss()

  for i in range(num_iter):
    #tz = model.forward(tx) #tensor(m)
    tz = model(tx) #tensor(m)
    cost = cost_func(tz, ty)
    print('i: %d, Cost: %f' % (i, cost.item()))
    cost.backward()
    opt.step()
    opt.zero_grad()

  return model

def evaluate_model(model, tx, ty):
  tz = model(tx)
  predict = (tz > 0.5).float()
  accuracy = (predict == ty).float().mean()
  return accuracy


In [ ]:
#Training the model
import torch.optim as optim

tx = torch.tensor(x_scaled, dtype=torch.float32)

models = []
for k in range(4):
  yk = np.copy(y)
  id_k = (y==k)
  id_notk = (y!=k)
  yk[id_k] = 1
  yk[id_notk] = 0
  tyk = torch.tensor(yk, dtype=torch.float32)

  print('\n\nTraining Model %d-------' % k)
  model = create_train_model(tx, tyk)
  models.append(model)
  accuracy = evaluate_model(model, tx, tyk)
  print('\nAccuracy: %f' % accuracy)




Training Model 0-------
i: 0, Cost: 0.924383
i: 1, Cost: 0.452481
i: 2, Cost: 0.281032
i: 3, Cost: 0.228103
i: 4, Cost: 0.179369
i: 5, Cost: 0.141593
i: 6, Cost: 0.115108
i: 7, Cost: 0.096658
i: 8, Cost: 0.083530
i: 9, Cost: 0.074138
i: 10, Cost: 0.067571
i: 11, Cost: 0.063195
i: 12, Cost: 0.060450
i: 13, Cost: 0.058803
i: 14, Cost: 0.057780
i: 15, Cost: 0.057021
i: 16, Cost: 0.056305
i: 17, Cost: 0.055529
i: 18, Cost: 0.054665
i: 19, Cost: 0.053721
i: 20, Cost: 0.052714
i: 21, Cost: 0.051668
i: 22, Cost: 0.050617
i: 23, Cost: 0.049606
i: 24, Cost: 0.048687
i: 25, Cost: 0.047910
i: 26, Cost: 0.047305
i: 27, Cost: 0.046883
i: 28, Cost: 0.046625
i: 29, Cost: 0.046491
i: 30, Cost: 0.046433
i: 31, Cost: 0.046400
i: 32, Cost: 0.046353
i: 33, Cost: 0.046268
i: 34, Cost: 0.046139
i: 35, Cost: 0.045976
i: 36, Cost: 0.045795
i: 37, Cost: 0.045614
i: 38, Cost: 0.045446
i: 39, Cost: 0.045297
i: 40, Cost: 0.045168
i: 41, Cost: 0.045051
i: 42, Cost: 0.044941
i: 43, Cost: 0.044833
i: 44, Cost: 0.0

In [ ]:
ty = torch.tensor(y, dtype=torch.long)

tz = [] #list(4) of tensor(m)
for k in range(4):
  tzk = models[k](tx)
  tz.append(tzk)

tz = torch.stack(tz, dim=1) #tensor(m, 4)
predict = torch.argmax(tz, dim=1) #tensor(m) of type long

accuracy = (predict == ty).float().mean()
print('\nOverall Accuracy: %f' % accuracy)



Overall Accuracy: 0.983333
